# MobileNetV2 + Transfer Learning + Dense + Softmax
## Klasifikasi Citra Medis Ultrasound Payudara (BUSI Dataset)
**Kategori:** Benign, Malignant, Normal

---
**Cara Pakai:**
1. Jalankan cell 1: download dataset dari Kaggle (via kagglehub)
2. Cell 2: restruktur dataset ke folder per kelas
3. Jalankan cell-cell berikutnya berurutan
4. Cell terakhir: download hasil training ke komputer

## 1. Download Dataset dari Kaggle

In [ ]:
# Install kagglehub (tanpa perlu upload API key)
!pip install -q kagglehub

import kagglehub
import os

print('Mendownload dataset BUSI dari Kaggle...')
path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
print(f'Dataset downloaded to: {path}')

# Copy ke folder Colab
!cp -r "$path" /content/BUSI_Dataset
print('Dataset siap di /content/BUSI_Dataset')

## 2. Restruktur Dataset ke Folder per Kelas

Dataset BUSI dari Kaggle menyimpan semua gambar di satu folder dengan nama file:
- `benign (1).png`, `benign (2).png`, ...
- `malignant (1).png`, `malignant (1).png`, ...
- `normal (1).png`, `normal (2).png`, ...

Cell ini akan memisahkan ke folder `benign/`, `malignant/`, `normal/`.

In [ ]:
import shutil
import glob
import re

BASE_PATH = '/content/BUSI_Dataset'

# Cek struktur dataset
all_items = os.listdir(BASE_PATH)
print('Isi folder:', all_items[:20], '...' if len(all_items) > 20 else '')

# Jika sudah ada subfolder benign/malignant/normal, lewati
class_folders = ['benign', 'malignant', 'normal']
already_structured = any(os.path.isdir(os.path.join(BASE_PATH, d)) for d in class_folders if d in all_items)

if already_structured:
    print('Dataset sudah terstruktur dengan folder per kelas.')
else:
    print('Dataset flat. Memulai restruktur ke folder per kelas...')
    
    # Cari folder Dataset_BUSI_with_GT jika ada
    src_dir = BASE_PATH
    for item in all_items:
        item_path = os.path.join(BASE_PATH, item)
        if os.path.isdir(item_path) and 'BUSI' in item.upper():
            src_dir = item_path
            print(f'Folder dataset ditemukan: {item_path}')
            break
    
    # Buat folder per kelas
    target_dirs = {}
    for cls in class_folders:
        d = os.path.join(BASE_PATH, cls)
        os.makedirs(d, exist_ok=True)
        target_dirs[cls] = d
    
    # Ambil semua file gambar
    image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    all_images = []
    for root, dirs, files in os.walk(src_dir):
        for f in files:
            if f.lower().endswith(image_extensions):
                all_images.append(os.path.join(root, f))
    
    print(f'Total gambar ditemukan: {len(all_images)}')
    
    # Pindahkan file berdasarkan prefix nama
    moved_count = 0
    mask_count = 0
    for img_path in all_images:
        fname = os.path.basename(img_path).lower()
        
        # Skip mask files
        if '_mask' in fname:
            mask_count += 1
            continue
        
        # Tentukan kelas dari prefix nama file
        if fname.startswith('benign'):
            target = target_dirs['benign']
        elif fname.startswith('malignant'):
            target = target_dirs['malignant']
        elif fname.startswith('normal'):
            target = target_dirs['normal']
        else:
            continue  # skip file tidak dikenal
        
        # Hindari duplikasi nama
        dest = os.path.join(target, fname)
        if os.path.exists(dest):
            base, ext = os.path.splitext(fname)
            dest = os.path.join(target, f'{base}_{moved_count}{ext}')
        
        shutil.move(img_path, dest)
        moved_count += 1
    
    print(f'Dipindahkan: {moved_count}')
    print(f'Mask files (diabaikan): {mask_count}')
    
    # Hapus folder kosong bekas
    if src_dir != BASE_PATH:
        shutil.rmtree(src_dir, ignore_errors=True)
        print(f'Folder {src_dir} dihapus.')

# Verifikasi akhir
print('\n=== Struktur final ===')
for cls in class_folders:
    cls_path = os.path.join(BASE_PATH, cls)
    if os.path.exists(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(image_extensions)])
        print(f'  {cls}/: {count} gambar')

print(f'\nDataset siap digunakan!')

## 3. Import Library

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import os
import time
import datetime

print('TensorFlow version:', tf.__version__)

## 4. Konfigurasi Parameter

In [ ]:
BASE_PATH = '/content/BUSI_Dataset'
OUTPUT_DIR = '/content/BUSI_Model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
EPOCHS_INITIAL = 20
EPOCHS_FINETUNE = 30
LEARNING_RATE_INITIAL = 1e-3
LEARNING_RATE_FINETUNE = 1e-5
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.1
RANDOM_SEED = 42
N_CLASSES = 3
CLASS_NAMES = ['benign', 'malignant', 'normal']

print('Base path:', BASE_PATH)
print('Output dir:', OUTPUT_DIR)
print('Classes:', CLASS_NAMES)

## 5. Data Augmentation & Preprocessing

In [ ]:
# Augmentasi untuk training
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT + TEST_SPLIT
)

# Tanpa augmentasi untuk validation/test
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VALIDATION_SPLIT + TEST_SPLIT
)

## 6. Load Dataset (Train / Val / Test)

In [ ]:
# Training set
train_generator = train_datagen.flow_from_directory(
    BASE_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=RANDOM_SEED
)

# Validation + Test (gabung dulu)
val_test_generator = val_test_datagen.flow_from_directory(
    BASE_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=RANDOM_SEED
)

## 7. Split Validation jadi Val + Test

In [ ]:
# Kumpulkan semua file dari val_test_generator
all_filenames = val_test_generator.filenames
all_labels_orig = val_test_generator.labels
n_total = len(all_filenames)
n_test = int(n_total * TEST_SPLIT / (VALIDATION_SPLIT + TEST_SPLIT))

# Shuffle indices
np.random.seed(RANDOM_SEED)
indices = np.arange(n_total)
np.random.shuffle(indices)

test_indices = indices[:n_test]
val_indices = indices[n_test:]

print(f'Total val+test: {n_total}')
print(f'Test samples: {len(test_indices)}')
print(f'Validation samples: {len(val_indices)}')
print(f'Train samples: {train_generator.samples}')

## 8. Buat Generator Val & Test (via DataFrame)

In [ ]:
def create_subset_generator(original_gen, indices):
    import pandas as pd
    # Ambil nama folder sebagai class label (string)
    class_labels = [all_filenames[i].split('/')[0] for i in indices]
    df = pd.DataFrame({
        'filename': [all_filenames[i] for i in indices],
        'class': class_labels
    })
    generator = val_test_datagen.flow_from_dataframe(
        df,
        directory=original_gen.directory,
        x_col='filename',
        y_col='class',
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    return generator

val_generator = create_subset_generator(val_test_generator, val_indices)
test_generator = create_subset_generator(val_test_generator, test_indices)

print('Validation samples:', val_generator.samples)
print('Test samples:', test_generator.samples)
print('Class mapping:', train_generator.class_indices)

## 9. Bangun Arsitektur Model

In [ ]:
# Load MobileNetV2 dengan pre-trained ImageNet weights
base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# Freeze base model (transfer learning phase)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(N_CLASSES, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

## 10. Compile Model (Phase 1 - Transfer Learning)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_INITIAL),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 11. Callbacks

In [ ]:
best_model_path = os.path.join(OUTPUT_DIR, 'best_model_phase1.keras')
final_model_path = os.path.join(OUTPUT_DIR, 'model.keras')
tensorboard_log_dir = os.path.join(OUTPUT_DIR, 'logs', datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        best_model_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    TensorBoard(
        log_dir=tensorboard_log_dir,
        histogram_freq=1
    )
]

print('Callbacks siap.')

## 12. Training Phase 1 (Feature Extraction)

In [ ]:
start_time = time.time()

history_phase1 = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=val_generator,
    validation_steps=val_generator.samples // BATCH_SIZE + 1,
    epochs=EPOCHS_INITIAL,
    callbacks=callbacks,
    verbose=1
)

elapsed = time.time() - start_time
print(f'Phase 1 training selesai dalam {elapsed:.2f} detik')

## 13. Fine Tuning (Phase 2)

In [ ]:
# Unfreeze base model
base_model.trainable = True

# Freeze layer awal, fine tune layer akhir
fine_tune_at = 120
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f'Fine tune from layer {fine_tune_at}')
print(f'Trainable layers: {len(base_model.layers) - fine_tune_at}')

In [ ]:
# Re-compile dengan learning rate lebih rendah
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_FINETUNE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
best_model_path_phase2 = os.path.join(OUTPUT_DIR, 'best_model_phase2.keras')

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        best_model_path_phase2,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    TensorBoard(
        log_dir=tensorboard_log_dir + '_finetune',
        histogram_freq=1
    )
]

In [ ]:
start_time = time.time()

history_phase2 = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=val_generator,
    validation_steps=val_generator.samples // BATCH_SIZE + 1,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks_phase2,
    verbose=1
)

elapsed = time.time() - start_time
print(f'Phase 2 training selesai dalam {elapsed:.2f} detik')

## 14. Evaluasi Model pada Data Test

In [ ]:
if os.path.exists(best_model_path_phase2):
    model.load_weights(best_model_path_phase2)
    print('Loaded best weights from phase 2')

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
test_generator.reset()
y_pred_prob = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_generator.classes[:len(y_pred)]

print('\n=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=list(test_generator.class_indices.keys()), zero_division=0))

acc = accuracy_score(y_true, y_pred)
print(f'Accuracy: {acc * 100:.2f}%')

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(test_generator.class_indices.keys()),
            yticklabels=list(test_generator.class_indices.keys()))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Confusion matrix saved.')

## 15. Plot Training History

In [ ]:
def combine_history(h1, h2):
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

full_history = combine_history(history_phase1, history_phase2)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(full_history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(full_history['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.axvline(x=len(history_phase1.history['accuracy']), color='gray', linestyle='--', alpha=0.5, label='Fine Tune Start')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(full_history['loss'], label='Train Loss', linewidth=2)
plt.plot(full_history['val_loss'], label='Val Loss', linewidth=2)
plt.axvline(x=len(history_phase1.history['loss']), color='gray', linestyle='--', alpha=0.5, label='Fine Tune Start')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150)
plt.show()
print('Training history plot saved.')

## 16. Simpan Model & Artifacts

In [ ]:
# Simpan model
model.save(final_model_path)
print(f'Model saved: {final_model_path}')

# Simpan class names
class_names_path = os.path.join(OUTPUT_DIR, 'class_names.json')
with open(class_names_path, 'w') as f:
    json.dump(list(test_generator.class_indices.keys()), f, indent=2)
print(f'Class names saved: {class_names_path}')

# Simpan history
history_path = os.path.join(OUTPUT_DIR, 'history.json')
with open(history_path, 'w') as f:
    json.dump(full_history, f, indent=2)
print(f'History saved: {history_path}')

# Simpan class indices
class_indices_path = os.path.join(OUTPUT_DIR, 'class_indices.json')
with open(class_indices_path, 'w') as f:
    json.dump(test_generator.class_indices, f, indent=2)
print(f'Class indices saved: {class_indices_path}')

print('\n=== Semua file telah disimpan ===')

## 17. Verifikasi Model

In [ ]:
loaded_model = keras.models.load_model(final_model_path)
print('Model berhasil di-load kembali.')

test_generator.reset()
x_test_sample, y_test_sample = next(test_generator)
predictions = loaded_model.predict(x_test_sample)

class_names_list = list(test_generator.class_indices.keys())
for i in range(min(3, len(x_test_sample))):
    pred_class = class_names_list[np.argmax(predictions[i])]
    confidence = np.max(predictions[i]) * 100
    print(f'Sample {i+1}: Predicted={pred_class}, Confidence={confidence:.2f}%')
    for j, cls in enumerate(class_names_list):
        print(f'  {cls}: {predictions[i][j] * 100:.2f}%')

## 18. Download Hasil ke Komputer Lokal

In [ ]:
from google.colab import files

output_files = [
    'model.keras',
    'class_names.json',
    'class_indices.json',
    'history.json',
    'confusion_matrix.png',
    'training_history.png'
]

for f in output_files:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        files.download(path)
        print(f'Downloaded: {f}')
    else:
        print(f'Not found: {f}')

print('\nSemua file selesai di-download.')

---
## Selesai

File yang dihasilkan:
- **model.keras** - Model final
- **class_names.json** - Daftar kelas
- **class_indices.json** - Mapping index ke kelas
- **history.json** - Riwayat training
- **confusion_matrix.png** - Confusion matrix
- **training_history.png** - Plot accuracy & loss